# FLAMES 30 Dor — review notebook

This notebook is a compact review of the work done so far on the **FLAMES 30 Dor** observations and their analysis.

## Main conclusions so far

- The current dataset being inspected is composed of **ADP FITS products**, not raw pipeline inputs.
- At least some files are **1D reduced spectra** stored as a FITS **binary table**:
  - `WAVE`
  - `FLUX_REDUCED`
  - `ERR_REDUCED`
  - `SNR`
- Therefore:
  - these files are good for **Python-based inspection and science analysis**
  - they are **not** the ideal input for testing the full GIRAFFE reduction chain
- Object labels like `t01` are **not globally unique sky positions**. The same label can appear at different pointings / coordinates.
- A physically meaningful comparison should use:
  - object name
  - coordinates
  - exposure-time family
  - and sometimes observing setup / provenance from the header

## Recommended workflow structure

Use one general notebook to build and enrich the master dataframe, then separate notebooks for each sample family:

- `00_build_master_dataframe.ipynb`
- `01_letter_sample.ipynb`
- `02_pos_sample.ipynb`
- `03_num_sample.ipynb`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.coordinates import Angle
from astropy.wcs import WCS
import astropy.units as u
import pickle

# Adjust these paths to your local setup
base_dir = Path(r"C:\Users\javas\Dropbox\GitHub\VIENTO\structure_function_analysis\FLAMES_30Dor")
data_folder = "giraffe_30Dor_ADP_unpacked"   # change if needed
fits_dir = base_dir / data_folder

print("Base dir:", base_dir)
print("FITS dir:", fits_dir)
print("Exists:", fits_dir.exists())

## 1. Build the master metadata dataframe

This was the first major step:
- scan all FITS files
- read primary headers
- extract object name, sky coordinates, observation info
- save the result as a dataframe

In [ ]:
def parse_ra_dec_from_header(hdr):
    ra = hdr.get("RA")
    dec = hdr.get("DEC")

    # Try numeric degrees first
    try:
        if ra is not None and dec is not None:
            return float(ra), float(dec)
    except Exception:
        pass

    # Try sexagesimal fallback
    try:
        if ra is not None and dec is not None:
            ra_deg = Angle(str(ra), unit=u.hourangle).degree
            dec_deg = Angle(str(dec), unit=u.deg).degree
            return ra_deg, dec_deg
    except Exception:
        pass

    return np.nan, np.nan


def read_basic_metadata(fits_path):
    with fits.open(fits_path) as hdul:
        hdr = hdul[0].header

    ra_deg, dec_deg = parse_ra_dec_from_header(hdr)

    row = {
        "filename": fits_path.name,
        "object": hdr.get("OBJECT"),
        "date_obs": hdr.get("DATE-OBS"),
        "mjd_obs": hdr.get("MJD-OBS"),
        "instrument": hdr.get("INSTRUME"),
        "telescope": hdr.get("TELESCOP"),
        "arcfile": hdr.get("ARCFILE"),
        "prog_id": hdr.get("HIERARCH ESO OBS PROG ID"),
        "obs_name": hdr.get("HIERARCH ESO OBS NAME"),
        "pro_catg": hdr.get("HIERARCH ESO PRO CATG"),
        "dpr_catg": hdr.get("HIERARCH ESO DPR CATG"),
        "dpr_type": hdr.get("HIERARCH ESO DPR TYPE"),
        "dpr_tech": hdr.get("HIERARCH ESO DPR TECH"),
        "ra_deg": ra_deg,
        "dec_deg": dec_deg,
    }
    return row


fits_files = sorted(fits_dir.glob("*.fits"))
rows = []
bad_files = []

for f in fits_files:
    try:
        rows.append(read_basic_metadata(f))
    except Exception as e:
        bad_files.append((f.name, str(e)))

df = pd.DataFrame(rows)

print(f"Read {len(df)} FITS files")
print("Bad files:", len(bad_files))
df.head()

## 2. Add useful header-derived columns

This is important to do in the **general notebook**, then re-export the dataframe.

Especially useful:
- `EXPTIME`
- `DATE-OBS`
- `HIERARCH ESO OBS NAME`
- `HIERARCH ESO PRO CATG`
- `HIERARCH ESO PRO REC1 RAW1 NAME`
- `HIERARCH ESO PRO REC1 PIPE ID`

In [ ]:
def add_header_keywords(df_in, fits_dir, key_map):
    df_out = df_in.copy()

    for newcol, hdrkey in key_map.items():
        values = []
        for fname in df_out["filename"]:
            fpath = fits_dir / fname
            try:
                with fits.open(fpath) as hdul:
                    hdr = hdul[0].header
                    values.append(hdr.get(hdrkey, np.nan))
            except Exception:
                values.append(np.nan)
        df_out[newcol] = values

    return df_out


key_map = {
    "EXPTIME": "EXPTIME",
    "OBS_NAME": "HIERARCH ESO OBS NAME",
    "PRO_CATG": "HIERARCH ESO PRO CATG",
    "RAW1_NAME": "HIERARCH ESO PRO REC1 RAW1 NAME",
    "PIPE_ID": "HIERARCH ESO PRO REC1 PIPE ID",
}

# Example:
# df = add_header_keywords(df, fits_dir, key_map)
# df.head()

## 3. Classify the object names

The object names were separated into three practical families:
- one letter + two digits, like `t01`, `u50`
- `pos` + digits, like `pos082`
- digits only

In [ ]:
obj = df["object"].astype(str).str.strip().str.lower()

conditions = [
    obj.str.fullmatch(r"[a-z]\d{2}"),
    obj.str.fullmatch(r"pos\d+"),
    obj.str.fullmatch(r"\d+"),
]

choices = [
    "letter_two_digits",
    "pos_digits",
    "digits_only",
]

df["object_group"] = np.select(conditions, choices, default="other")

df_letter = df[df["object_group"] == "letter_two_digits"].copy()
df_pos    = df[df["object_group"] == "pos_digits"].copy()
df_num    = df[df["object_group"] == "digits_only"].copy()

print(df["object_group"].value_counts())

## 4. Export and reload the dataframes

This was used to keep one general notebook for building the metadata and separate notebooks for each family.

In [ ]:
pickle_path = base_dir / "flames30dor_dataframes.pkl"

df_dict = {
    "df": df,
    "df_letter": df_letter,
    "df_pos": df_pos,
    "df_num": df_num,
}

# Save
# with open(pickle_path, "wb") as f:
#     pickle.dump(df_dict, f)

# Reload
# with open(pickle_path, "rb") as f:
#     df_dict = pickle.load(f)

# df = df_dict["df"]
# df_letter = df_dict["df_letter"]
# df_pos = df_dict["df_pos"]
# df_num = df_dict["df_num"]

## 5. Plot the sky positions

This gave the first global visual inspection of the observations.

### Simple scatter version

In [ ]:
df_plot = df_letter.dropna(subset=["ra_deg", "dec_deg"]).copy()

plt.figure(figsize=(10, 7))
plt.scatter(df_plot["ra_deg"], df_plot["dec_deg"], s=20)

for _, row in df_plot.iterrows():
    label = row["object"] if pd.notna(row["object"]) and row["object"] else row["filename"]
    plt.text(row["ra_deg"], row["dec_deg"], label, fontsize=7)

plt.xlabel("RA [deg]")
plt.ylabel("Dec [deg]")
plt.title("Sky positions of df_letter")
plt.gca().invert_xaxis()
plt.grid(True, alpha=0.3)
plt.show()

### WCSAxes version

A custom celestial WCS was built for plotting positions in a world-coordinate frame. This is useful even if the ADP files are not image products.

In [ ]:
ra_center = df_plot["ra_deg"].mean()
dec_center = df_plot["dec_deg"].mean()

wcs = WCS(naxis=2)
wcs.wcs.crpix = [500, 500]
wcs.wcs.cdelt = np.array([-0.01, 0.01])   # deg/pixel
wcs.wcs.crval = [ra_center, dec_center]
wcs.wcs.ctype = ["RA---TAN", "DEC--TAN"]

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(1, 1, 1, projection=wcs)

ax.scatter(
    df_plot["ra_deg"],
    df_plot["dec_deg"],
    s=20,
    transform=ax.get_transform("world")
)

for _, row in df_plot.iterrows():
    label = row["object"] if pd.notna(row["object"]) and row["object"] else row["filename"]
    ax.text(
        row["ra_deg"],
        row["dec_deg"],
        label,
        fontsize=7,
        transform=ax.get_transform("world")
    )

ax.set_title("Sky positions of df_letter")
ax.coords[0].set_axislabel("Right Ascension")
ax.coords[1].set_axislabel("Declination")
ax.grid(color="gray", alpha=0.3, linestyle="--")

plt.show()

## 6. Inspect one FITS file structurally

A key discovery was that some ADP files are **not images**, but **1D reduced spectra stored as a binary table**.

In [ ]:
example_file = fits_dir / "ADP.2019-09-04T13_34_26.413.fits"  # adjust if needed

with fits.open(example_file) as hdul:
    print(hdul.info())
    print(hdul[1].columns)

Typical result found in the analysis:

- `PRIMARY` HDU with no image data
- `SPECTRUM` extension as a `BinTableHDU`
- columns:
  - `WAVE` [nm]
  - `FLUX_REDUCED` [adu]
  - `ERR_REDUCED` [adu]
  - `SNR`

This explained why DS9 could fail to open the file as a normal image: it is a spectral table product, not a 2D image.

## 7. Load all spectra from a dataframe

This became the main pattern for loading spectra from a subset dataframe such as `df_letter`.

In [ ]:
def load_spectra_from_df(df_in, fits_dir, flux_col="FLUX_REDUCED", err_col="ERR_REDUCED"):
    spectra = []

    for _, row in df_in.iterrows():
        fits_path = fits_dir / row["filename"]

        try:
            with fits.open(fits_path) as hdul:
                spec = hdul[1].data

                wave = spec["WAVE"][0]
                flux = spec[flux_col][0]
                err  = spec[err_col][0]
                snr  = spec["SNR"][0] if "SNR" in spec.names else np.nan

                spectra.append({
                    "filename": row["filename"],
                    "object": row["object"],
                    "ra_deg": row["ra_deg"],
                    "dec_deg": row["dec_deg"],
                    "wave": wave,
                    "flux": flux,
                    "err": err,
                    "snr": snr,
                })

        except Exception as e:
            print(f"Could not read {fits_path.name}: {e}")

    print(f"Loaded {len(spectra)} spectra")
    return spectra


# Example:
# letter_spectra = load_spectra_from_df(df_letter, fits_dir)

## 8. Plot spectra in a wavelength window

This was the main inspection pattern for zooming into lines like Hα and [N II].

In [ ]:
def plot_spectra_subset(spectra_list, wmin, wmax, normalize=False):
    plt.figure(figsize=(12, 6))

    for spec in spectra_list:
        wave = spec["wave"]
        flux = spec["flux"]
        mask = (wave >= wmin) & (wave <= wmax)

        wave_zoom = wave[mask]
        flux_zoom = flux[mask]

        if len(wave_zoom) == 0:
            continue

        if normalize and np.nanmax(flux_zoom) != 0:
            flux_zoom = flux_zoom / np.nanmax(flux_zoom)

        plt.plot(wave_zoom, flux_zoom, alpha=0.7, lw=1)

    plt.xlabel("Wavelength [nm]")
    plt.ylabel("Normalized Flux" if normalize else "Reduced Flux [adu]")
    plt.title(f"Spectra zoom: {wmin}–{wmax} nm")
    plt.grid(True, alpha=0.3)
    plt.show()


# Example:
# letter_spectra = load_spectra_from_df(df_letter, fits_dir)
# plot_spectra_subset(letter_spectra, 655, 658, normalize=False)
# plot_spectra_subset(letter_spectra, 655, 658, normalize=True)

### Export one PNG per spectrum instead of plotting everything in Jupyter

This became the practical solution when many spectra had to be reviewed.

In [ ]:
def save_zoom_plots(spectra_list, outdir, wmin, wmax):
    outdir = Path(outdir)
    outdir.mkdir(exist_ok=True)

    for spec in spectra_list:
        wave = spec["wave"]
        flux = spec["flux"]
        mask = (wave >= wmin) & (wave <= wmax)

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(wave[mask], flux[mask])
        ax.set_xlabel("Wavelength [nm]")
        ax.set_ylabel("Reduced Flux [adu]")
        ax.set_title(f"{spec['object']} | {spec['filename']} | {wmin}–{wmax} nm")
        ax.grid(True, alpha=0.3)

        obj = str(spec["object"]).replace("/", "_").replace("\\", "_").replace(" ", "_")
        base = Path(spec["filename"]).stem
        fname = outdir / f"{obj}_{base}_{wmin}_{wmax}.png"

        fig.savefig(fname, dpi=200, bbox_inches="tight")
        plt.close(fig)

    print(f"Saved {len(spectra_list)} plots in: {outdir.resolve()}")


# Example:
# save_zoom_plots(letter_spectra, "plots_letter_zoom", 655, 658)

## 9. Compare repeated spectra of the same object label

A key question was: why do multiple files exist for labels like `t01`?

The answer found in the headers was:
- the same object label can appear at **different coordinates**
- the same label can also appear at different pointings / observing setups
- the same label can appear with different exposure times or provenance

In [ ]:
keys = [
    "OBJECT",
    "DATE",
    "DATE-OBS",
    "MJD-OBS",
    "EXPTIME",
    "INSTRUME",
    "TELESCOP",
    "RA",
    "DEC",
    "ARCFILE",
    "HIERARCH ESO OBS PROG ID",
    "HIERARCH ESO OBS NAME",
    "HIERARCH ESO DPR CATG",
    "HIERARCH ESO DPR TYPE",
    "HIERARCH ESO DPR TECH",
    "HIERARCH ESO PRO CATG",
    "HIERARCH ESO PRO TECH",
    "HIERARCH ESO PRO REC1 ID",
    "HIERARCH ESO PRO REC1 PIPE ID",
    "HIERARCH ESO PRO REC1 RAW1 NAME",
]

def build_header_table(df_subset, fits_dir, keys):
    rows = []

    for _, row in df_subset.iterrows():
        fpath = fits_dir / row["filename"]
        try:
            with fits.open(fpath) as hdul:
                hdr = hdul[0].header

            out = {"filename": row["filename"], "object": row["object"]}
            for k in keys:
                out[k] = hdr.get(k)
            rows.append(out)

        except Exception as e:
            rows.append({
                "filename": row["filename"],
                "object": row["object"],
                "ERROR": str(e)
            })

    return pd.DataFrame(rows)

def varying_columns(df_in):
    return [
        col for col in df_in.columns
        if df_in[col].astype(str).nunique(dropna=False) > 1
    ]

# Example: inspect one group like t01
# df_t01 = df_letter[df_letter["object"].astype(str).str.strip().str.lower() == "t01"].copy()
# df_t01_hdr = build_header_table(df_t01, fits_dir, keys)
# print(df_t01_hdr[varying_columns(df_t01_hdr)])

### Important interpretation from the `t01` case

One important result from the conversation was that `t01` was **not one unique sky position**. It split across multiple pointings and coordinates.

This means:
- comparing all `t01` spectra together can be misleading
- more meaningful comparisons should use:
  - object name
  - coordinates
  - exposure-time family
  - and sometimes observing-pointing name (`OBS NAME`)

## 10. Group by exposure-time family

The raw `EXPTIME` values were very close but not numerically identical, for example:

- `1199.9699`
- `1199.9741`
- `1199.9691`

So exact `EXPTIME` was too strict. The practical solution was to create an exposure-time family by rounding.

In [ ]:
df_letter["EXPTIME_group"] = df_letter["EXPTIME"].round(0)

print(df_letter[["EXPTIME", "EXPTIME_group"]].drop_duplicates().sort_values("EXPTIME"))
print(df_letter["EXPTIME_group"].value_counts().sort_index())

## 11. Subset one exposure-time family and inspect duplicates

This was the simpler approach before building more complex sample identifiers:
- choose one exposure-time family
- inspect repeated coordinates or repeated `object + position`

In [ ]:
# Example: 1200 s family
df_exp1200 = df_letter[df_letter["EXPTIME_group"] == 1200].copy()

df_exp1200["ra_round"] = df_exp1200["ra_deg"].round(6)
df_exp1200["dec_round"] = df_exp1200["dec_deg"].round(6)

# Repeated object + coordinates
dup_obj_pos = df_exp1200[
    df_exp1200.duplicated(subset=["object", "ra_round", "dec_round"], keep=False)
]

print(
    dup_obj_pos[["object", "ra_round", "dec_round", "filename"]]
    .sort_values(["object", "ra_round", "dec_round", "filename"])
)

## 12. Build a sample identifier later if needed

Once the structure is understood, a more physically meaningful sample ID can be created from:
- object name
- rounded RA
- rounded Dec
- grouped exposure time

In [ ]:
df_letter["object_clean"] = df_letter["object"].astype(str).str.strip().str.lower()

df_letter["sample_id"] = (
    df_letter["object_clean"]
    + "_ra" + df_letter["ra_round"].astype(str)
    + "_dec" + df_letter["dec_round"].astype(str)
    + "_exp" + df_letter["EXPTIME_group"].astype(int).astype(str)
)

sample_unique = (
    df_letter[
        ["object", "ra_round", "dec_round", "EXPTIME_group", "sample_id"]
    ]
    .drop_duplicates()
    .sort_values(["object", "ra_round", "dec_round", "EXPTIME_group"])
)

print(sample_unique.head(30))

## Final interpretation

### What is being analyzed right now
The current FLAMES 30 Dor dataset consists of **ADP processed spectral products**, not raw pipeline inputs.

### What these files are good for
- inspect headers
- inspect sky distribution of observations
- load and compare reduced spectra
- group targets by object family, position, and exposure-time family
- study repeated observations and provenance

### What these files are not ideal for
- testing the full reduction pipeline from raw frames

For that, a separate raw-data branch is needed with:
- raw GIRAFFE/FLAMES-UVES files
- associated calibrations
- EDPS / pipeline workflow inputs

### Most important conceptual lesson
Object labels like `t01` are **not globally unique observations**. Analysis should be done with at least:
- object
- position
- exposure-time family